In [ ]:
import pystac_client
import odc.stac
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
# -----------------------
# USER INPUT
# -----------------------
lat = 37.97
lon = 23.73
buffer = 0.01
time_range = "2025-01-01/2026-04-01"
bbox = [lon-buffer, lat-buffer, lon+buffer, lat+buffer]

In [ ]:
# -----------------------
# STAC SEARCH
# -----------------------
catalog = pystac_client.Client.open(
    "https://satellite.cityofathens.gr/pycsw/stac"
)

search = catalog.search(
    collections=["landsat-c2-l2-ath"],
    bbox=bbox,
    datetime=time_range
)

items = list(search.items())

In [ ]:
# -----------------------
# LOAD DATA
# -----------------------
ds = odc.stac.load(
    items,
    bands=["lwir11"],
    bbox=bbox,
    resolution=30,
    groupby="solar_day"
)

lst = ds.lwir11

In [ ]:
# -----------------------
# SCALE TEMPERATURE
# -----------------------
lst_c = lst * 0.00341802 + 149.0 - 273.15
lst_c = lst_c.where((lst_c > -20) & (lst_c < 70))

In [ ]:
# -----------------------
# EXTRACT POINT
# -----------------------
point = lst_c.sel(
    x=lon,
    y=lat,
    method="nearest"
)

df = point.to_dataframe(name="temperature").reset_index()
df = df.dropna()

In [ ]:
# -----------------------
# STATISTICAL OUTLIERS
# -----------------------
mean = df["temperature"].mean()
std = df["temperature"].std()

df = df[(df["temperature"] > mean - 3*std) &
        (df["temperature"] < mean + 3*std)]

In [ ]:
# -----------------------
# INTERPOLATION
# -----------------------
df_obs = df.copy()
df_obs = df_obs.set_index("time")
df_interp = df.copy()
df_interp = df_interp.set_index("time")
df_interp = df_interp.resample("D").mean()
df_missing = df_interp["temperature"].isna()
df_interp["temperature"] = df_interp["temperature"].interpolate(
    method="time",
    limit=10
)
#df["temperature"] = df["temperature"].interpolate(method="linear")
#df["temperature"] = df["temperature"].interpolate(method="time", limit=10)
#df["temperature"] = df["temperature"].interpolate(method="spline", order=2)
#df = df.reset_index()

In [ ]:
# -----------------------
# PLOT
# -----------------------
# plt.figure(figsize=(15,8))
# plt.plot(
#     df["time"],
#     df["temperature"],
#     marker="o"
# )
# plt.xlabel("Date")
# plt.ylabel("Temperature (°C)")
# plt.title(f"Landsat Surface Temperature ({lat}, {lon})")
# plt.grid(True)
# # plt.savefig("/home/kalxas/landsat.png")
# plt.show()

In [ ]:
# -----------------------
# PLOT
# -----------------------
plt.figure(figsize=(15,8))

# interpolated smooth line
plt.plot(
    df_interp.index,
    df_interp["temperature"],
    color="red",
    linewidth=2,
    label="Interpolated temperature"
)

# real Landsat observations
plt.scatter(
    df_obs.index,
    df_obs["temperature"],
    color="black",
    s=40,
    label="Landsat observations"
)

# shade interpolated periods
plt.fill_between(
    df_interp.index,
    df_interp["temperature"],
    where=df_missing,
    color="orange",
    alpha=0.25,
    label="Interpolated period"
)

plt.xlabel("Date")
plt.ylabel("Temperature (°C)")
plt.title(f"Landsat Surface Temperature ({lat}, {lon})")
plt.grid(True)
plt.legend()

# plt.savefig("lst_timeseries.png")
plt.show()